# ocean-curation-pipeline-toolkit: Pipeline Walkthrough

**Purpose:** Demonstrate the full data packaging pipeline that transforms raw, messy oceanographic data files into archive-ready, FAIR-compliant submission packages with ISO 19115-2 metadata.

**Context:** This notebook walks through each pipeline step using sample data that simulates a GOMECC-4 Gulf of Mexico cruise. The sample intentionally includes common data quality issues that GRIIDC curators encounter in real submissions: mixed date formats, whitespace artifacts, missing values, duplicate rows, and non-standard headers.

**Author:** Ranjith Guggilla | Harte Research Institute, TAMU-CC

---

## Setup for Google Colab

If running in Google Colab, uncomment and run the cell below to clone the repository:

In [ ]:
# UNCOMMENT if running in Google Colab:
# !git clone https://github.com/ranjithguggilla/ocean-curation-pipeline-toolkit.git
# import os
# os.chdir('ocean-curation-pipeline-toolkit')

## 1. Environment Setup

In [ ]:
import os
import sys
import json
import yaml
from pathlib import Path

# Install required packages
print("Installing dependencies...")
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "netCDF4", "xarray"], check=False)

import pandas as pd
from lxml import etree
import netCDF4 as nc

print("✓ All imports successful\n")

# Resolve paths for both local and Colab environments
repo_path = Path.cwd()
if repo_path.name == 'notebooks':
    # Running from notebooks/ directory locally
    repo_path = repo_path.parent
elif 'ocean-curation-pipeline-toolkit' not in str(repo_path):
    # Might be in a different directory, try to find the repo
    if (Path.cwd().parent / 'sample_data').exists():
        repo_path = Path.cwd().parent

os.chdir(repo_path)
print(f"Working directory: {repo_path}\n")

# Show the project structure
result = subprocess.run(["find", ".", "-maxdepth", "2", "-type", "f", 
                        "-not", "-path", "./.git/*",
                        "-not", "-path", "./__pycache__/*",
                        "-not", "-path", "./output/*",
                        "-not", "-path", "./.pytest_cache/*",
                        "-not", "-name", "*.pyc"],
                       capture_output=True, text=True)
print("Project Structure:")
for line in sorted(result.stdout.strip().split("\n")):
    if line and "egg-info" not in line and "venv" not in line:
        print(f"  {line}")

## 2. Examine the Raw Data

This is what a PI might actually submit — a messy Excel file with common problems.

In [ ]:
# Load the messy Excel file
excel_path = repo_path / "sample_data/raw/GOMECC4_cruise_watersamples_RAW.xlsx"
raw_excel = pd.read_excel(excel_path, sheet_name="Water Chemistry")

print(f"Shape: {raw_excel.shape}")
print(f"\nColumn names (note inconsistencies):")
for i, col in enumerate(raw_excel.columns):
    print(f"  [{i}] '{col}'")

print(f"\n--- First 5 rows ---")
raw_excel.head()

In [ ]:
# Spot the problems a curator would catch
print("=== DATA QUALITY ISSUES ===")
print()

# 1. Mixed date formats
print("1. MIXED DATE FORMATS:")
dates = raw_excel["Date/Time"].dropna().unique()[:8]
for d in dates:
    print(f"   '{d}'")

# 2. Whitespace in station IDs
print("\n2. WHITESPACE IN STATION IDS:")
stations = raw_excel["Station  ID"].unique()
for s in stations[:5]:
    print(f"   '{s}' (len={len(str(s))})")

# 3. Missing values
print("\n3. MISSING VALUES:")
missing = raw_excel.isnull().sum()
for col, count in missing.items():
    if count > 0:
        print(f"   {col}: {count} missing ({count/len(raw_excel)*100:.1f}%)")

# 4. Duplicate rows
dups = raw_excel.duplicated().sum()
print(f"\n4. DUPLICATE ROWS: {dups}")

# 5. Empty rows
empty = raw_excel.isnull().all(axis=1).sum()
print(f"\n5. COMPLETELY EMPTY ROWS: {empty}")

## 3. Run the Full Pipeline

The pipeline executes 8 sequential steps: `init → validate → transform → profile → checksum → metadata → netcdf → package`

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "griidc_pack", "-c", "config.yaml", "run-all"],
    capture_output=True, text=True, cwd=str(repo_path)
)
print(result.stdout)
if result.returncode != 0 and result.stderr:
    print("\nWARNINGS/ERRORS:")
    print(result.stderr[:1000])

## 4. Inspect Validation Results

The validator detected issues in the raw data — exactly what a curator needs to see.

In [ ]:
validation_report = repo_path / "output/submission_v1.0.0/logs/validation_report.json"

with open(validation_report) as f:
    report = json.load(f)

print(f"Overall: {report['summary']}")
print(f"All passed: {report['all_passed']}\n")

for file_result in report["files"]:
    print(f"--- {file_result['file']} ---")
    for check in file_result["checks"]:
        status = "✓" if check["passed"] else "✗"
        print(f"  {status} {check['check']}: {check['detail']}")
    print()

## 5. Review Transformations

Every change is logged with full provenance — essential for scientific reproducibility.

In [ ]:
transform_summary = repo_path / "output/submission_v1.0.0/logs/transform_summary.json"

with open(transform_summary) as f:
    transforms = json.load(f)

for t in transforms:
    print(f"File: {t['input_file']} → {t.get('output_file', 'N/A')}")
    print(f"  Output: {t.get('output_rows', '?')} rows, {t.get('output_columns', '?')} cols")
    for step in t["transformations"]:
        action = step["action"]
        if action == "normalize_headers" and "mapping" in step:
            print(f"  {action}:")
            for old, new in step["mapping"].items():
                if old != new:
                    print(f"    '{old}' → '{new}'")
        else:
            print(f"  {action}: {json.dumps({k:v for k,v in step.items() if k != 'action'})}")
    print()

## 6. Before vs. After Comparison

In [ ]:
processed_csv = repo_path / "output/submission_v1.0.0/processed/gomecc4_cruise_watersamples_raw.csv"
processed = pd.read_csv(processed_csv)

print(f"Raw:       {raw_excel.shape[0]} rows, {raw_excel.shape[1]} cols")
print(f"Processed: {processed.shape[0]} rows, {processed.shape[1]} cols")
print(f"\nProcessed headers (normalized):")
for col in processed.columns:
    print(f"  {col}")

print(f"\n--- First 5 rows (cleaned) ---")
processed.head()

## 7. Data Quality Profile

Statistical summary with outlier detection — the kind of report curators write for every submission.

In [ ]:
quality_profile = repo_path / "output/submission_v1.0.0/logs/data_quality_profile.json"

with open(quality_profile) as f:
    profiles = json.load(f)

for p in profiles:
    print(f"=== {p['file']} ===")
    print(f"  Quality Grade: {p['quality_grade']}")
    print(f"  Completeness: {p['completeness_pct']}%")
    print(f"  Duplicate rows: {p['duplicate_rows']}")
    if p.get("spatial_coverage"):
        sc = p["spatial_coverage"]
        print(f"  Spatial: [{sc['south_lat']}°N..{sc['north_lat']}°N] × [{sc['west_lon']}°E..{sc['east_lon']}°E]")
    print(f"  Variables:")
    for var, stats in p["variables"].items():
        if stats["type"] == "numeric":
            out = f" ({stats.get('outliers_iqr',0)} outliers)" if stats.get('outliers_iqr',0) > 0 else ""
            print(f"    {var}: [{stats.get('min','?')}..{stats.get('max','?')}] μ={stats.get('mean','?')}{out}")
    print()

## 8. ISO 19115-2 Metadata

The generated XML follows the ISO 19115-2:2009 standard for geospatial metadata.

In [ ]:
# Parse and display key metadata elements
xml_path = repo_path / "output/submission_v1.0.0/metadata/iso19115-2.xml"
tree = etree.parse(str(xml_path))
root = tree.getroot()

ns = {
    "gmd": "http://www.isotc211.org/2005/gmd",
    "gco": "http://www.isotc211.org/2005/gco",
    "gml": "http://www.opengis.net/gml/3.2",
}

def get_text(xpath):
    el = root.find(xpath, ns)
    return el.text if el is not None else "N/A"

print("=== ISO 19115-2 Metadata Summary ===")
print()
print(f"File Identifier: {get_text('.//gmd:fileIdentifier/gco:CharacterString')}")
print(f"Title: {get_text('.//gmd:title/gco:CharacterString')}")
print(f"Date: {get_text('.//gmd:dateStamp/gco:Date')}")
print(f"Standard: {get_text('.//gmd:metadataStandardName/gco:CharacterString')}")

# Bounding box
bbox = root.find('.//gmd:EX_GeographicBoundingBox', ns)
if bbox is not None:
    w = get_text('.//gmd:westBoundLongitude/gco:Decimal')
    e = get_text('.//gmd:eastBoundLongitude/gco:Decimal')
    s = get_text('.//gmd:southBoundLatitude/gco:Decimal')
    n = get_text('.//gmd:northBoundLatitude/gco:Decimal')
    print(f"Bounding Box: [{s}°N, {n}°N] × [{w}°E, {e}°E]")

# Keywords
kws = root.findall('.//gmd:keyword/gco:CharacterString', ns)
print(f"\nKeywords ({len(kws)}):")
for kw in kws[:6]:
    print(f"  • {kw.text}")

print(f"\nXML file size: {xml_path.stat().st_size:,} bytes")

## 9. NetCDF-4 Output (CF-1.8 Compliant)

The pipeline exports to self-describing NetCDF with CF standard names — the preferred format for oceanographic data archives.

In [ ]:
nc_path = repo_path / "output/submission_v1.0.0/netcdf/gomecc4_cruise_watersamples_raw.nc"
ds = nc.Dataset(str(nc_path))

print(f"Format: {ds.data_model}")
print(f"Conventions: {ds.Conventions}")
print(f"Feature Type: {ds.featureType}")
print(f"Observations: {len(ds.dimensions['obs'])}\n")

print("CF-Mapped Variables:")
for vname in ds.variables:
    v = ds.variables[vname]
    sn = getattr(v, 'standard_name', None)
    if sn:
        units = getattr(v, 'units', '—')
        print(f"  {vname}: {sn} [{units}]")

ds.close()

## 10. SHA-256 Fixity Checksums

Every file in the package has a SHA-256 hash for long-term integrity verification.

In [ ]:
checksum_file = repo_path / "output/submission_v1.0.0/checksums.sha256"

with open(checksum_file) as f:
    print(f.read())

## 11. FAIR Compliance Audit

The package self-evaluates against all 15 FAIR sub-principles.

In [ ]:
fair_audit = repo_path / "output/submission_v1.0.0/FAIR_AUDIT.md"

with open(fair_audit) as f:
    print(f.read())

## 12. Final Submission Package

The complete deliverable — ready for GRIIDC catalog submission.

In [ ]:
result = subprocess.run(
    ["find", str(repo_path / "output/submission_v1.0.0"), "-type", "f"],
    capture_output=True, text=True
)
files = sorted(result.stdout.strip().split("\n"))
print(f"Package contents ({len(files)} files):\n")
for f in files:
    if f:  # Skip empty lines
        fpath = Path(f)
        size = fpath.stat().st_size
        rel = fpath.relative_to(repo_path / "output/submission_v1.0.0")
        print(f"  {str(rel): <55} {size:>8,} bytes")

archive = repo_path / "output/submission_v1.0.0.tar.gz"
if archive.exists():
    print(f"\nCompressed archive: {archive.stat().st_size:,} bytes")

## 13. Full Provenance Chain

Every action is recorded in JSON Lines format — complete audit trail.

In [ ]:
provenance_file = repo_path / "output/submission_v1.0.0/logs/provenance.jsonl"

with open(provenance_file) as f:
    for line in f:
        entry = json.loads(line)
        ts = entry.get("timestamp", "")
        action = entry.get("action", "unknown")
        # Compact display
        detail = {k: v for k, v in entry.items() if k not in ("timestamp", "action")}
        detail_str = json.dumps(detail, default=str)
        if len(detail_str) > 100:
            detail_str = detail_str[:100] + "..."
        print(f"[{ts}] {action}: {detail_str}")